# 下一季度指引预期配对核查

逐条审阅全部已下载报道，保留未知。金额必须在原段落出现；范围中点只从明确上下界计算；不由方向或百分比反推缺失预期。NVIDIA目标季度及可用指引数值与原始公司披露交叉核对。当前阶段不查看收益、不训练、不回测。

In [1]:
from pathlib import Path
import os,json,hashlib
os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
import qlib,pandas as pd,yaml
from qlib.workflow import R
from src.workflow.minute_action_value import _source
from src.data.issuer_guidance import read_guidance_facts
from src.data.guidance_review import validate_guidance_reviews
from src.data.nas_market_data import archive_nas_recorder
root=Path('/Users/rziiiii/dev/workspace');card=root/'research/guidance_consensus_review_20261014';plan=json.loads((card/'plan.json').read_text())
assert hashlib.sha256((card/'reviewed_facts.json').read_bytes()).hexdigest()==plan['reviewed_facts_sha256']
p=root/plan['issuer_fact_manifest'];assert hashlib.sha256(p.read_bytes()).hexdigest()==plan['issuer_fact_manifest_sha256']
qlib.init(provider_uri=str(root/plan['provider']),region='us',kernels=1)
objects,source_audit=_source(root,plan['source'],['sources/manifest'])
records,issuer=read_guidance_facts(root,json.loads(p.read_text()),as_of=plan['as_of'])
reviewed=json.loads((card/'reviewed_facts.json').read_text());frame=validate_guidance_reviews(objects['sources/manifest'],reviewed,issuer)
audit={'calendar_events':len(frame),'status_counts':frame.status.value_counts().to_dict(),'quoted_numeric_pairs':int(frame.numeric_pair_present.sum()),'known_directions':int(frame.comparison_direction_known.sum()),'nvidia_issuer_anchors':int(frame.issuer_target_anchor_checked.fillna(False).sum()),'economic_scope_caveat_rows':int(frame.scope_caveat.fillna(False).sum()),'market_return_series_loaded':False,'model_fits':0,'backtests':0,'normalized_economic_scope_certified':False,'first_public_timestamp_certified':False}
tracking=card/'qlib_recorder'
with R.start(experiment_name=plan['experiment'],recorder_name='prospective_quarterly_guidance_semantic_review',uri=str(tracking)):
    rec=R.get_recorder();rec.log_params(plan_sha256=hashlib.sha256((card/'plan.json').read_bytes()).hexdigest(),model_fits=0,backtests=0)
    rec.save_objects(artifact_path='inputs',source_audit=source_audit,source_manifest=objects['sources/manifest'],reviewed=reviewed,issuer_guidance=issuer,issuer_records=records)
    rec.save_objects(artifact_path='analysis',review=frame,audit=audit)
    rec.save_objects(artifact_path='sources',validation_code=(root/'src/data/guidance_review.py').read_text())
    rec.log_metrics(registered_artifact_count=8,reviewed_events=len(frame))
summary={'experiment':plan['experiment'],'decision':'quoted_guidance_comparisons_reviewed_conditional_research_only','source_audit':audit,'recorder_id':rec.id,'experiment_id':str(rec.experiment_id),'recorder_uri':str(tracking),'recorder_local_dir':str(tracking/str(rec.experiment_id)/rec.id),'provider_uri':str(root/plan['provider']),'region':'us','as_of':plan['as_of'],'qlib_docs':'https://qlib.readthedocs.io/en/latest/component/workflow.html','objective_achieved':False,'goal_remains_active':True,'promotion_eligible':False,'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/*','sources/*']}}
artifacts=tracking/str(rec.experiment_id)/rec.id/'artifacts';(card/'verification_manifest.json').write_text(json.dumps({'recorder_id':rec.id,'entries':[{'path':str(p.relative_to(card)),'sha256':hashlib.sha256(p.read_bytes()).hexdigest(),'bytes':p.stat().st_size} for p in sorted(artifacts.rglob('*')) if p.is_file()]},indent=2)+'\n')
summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=str(rec.experiment_id))
(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n');(card/'workflow.yaml').write_text(yaml.safe_dump({'experiment':plan['experiment'],'provider_uri':summary['provider_uri'],'recorder_uri':str(tracking),'artifact_policy':summary['artifact_policy']}));frame.to_csv(card/'review_summary.csv',index=False)
audit


{'calendar_events': 105,
 'status_counts': {'numeric_pair': 63,
  'unknown': 23,
  'direction_only': 15,
  'scope_caveat': 3,
  'source_missing': 1},
 'quoted_numeric_pairs': 66,
 'known_directions': 81,
 'nvidia_issuer_anchors': 32,
 'economic_scope_caveat_rows': 3,
 'market_return_series_loaded': False,
 'model_fits': 0,
 'backtests': 0,
 'normalized_economic_scope_certified': False,
 'first_public_timestamp_certified': False}

# 下一季度指引与预期配对核查

逐条审阅104份晨报，保留全部105个日历事件。66条有公开的指引/预期金额，其中3条带并购或中国收入排除的口径事项；另15条仅有明确比较方向。其余23条保持未知，1份原文件404。状态为63个普通数值配对、15个方向配对、3个口径事项、23个未知、1个缺源。

金额均在对应原段落中验证，11个范围中点只从明确上下界计算。没有把“高于50%”反推缺失分母；没有把全年、分部、EPS、同比或环比比较混为下一季度整体营收比较。“投资者失望”也不自动等于低于分析师一致预期。NVIDIA32个可核对事件的目标季度与已冻结公司指引一致，可用的指引金额也全部一致。

reviewed_facts.json保存助手逐条语义审阅输入及段落SHA，review_summary.csv保存校验结果，原文和原始公司事实保存在Qlib Recorder。本轮没有加载市场收益序列，没有按新闻中的股价反应选方向。金额的文本出现检查不能替代语义判断；这份审阅也不是独立人工双审或历史首次公开认证。

三处口径事项为Mellanox并购纳入、两处中国收入排除。它们按原文保留，不声称已完成同口径经济归因。发布时间与传输限制继承来源卡，promotion_eligible保持false。

本卡0模型、0回测，无新增收益。完成的数据审阅用于另立固定信息对照；不能把66个金额配对或81个已知方向当成稳定超额证据。目标仍active。
